In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain.llms import HuggingFacePipeline
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from fuzzywuzzy import fuzz
import torch
import json
from langchain.output_parsers import RegexParser
from langchain.schema import OutputParserException
from tqdm import tqdm
import re
from word2number import w2n
from src.shared_prompt import SYSTEM_PROMPT
from rapidfuzz import fuzz
from typing import List

ModuleNotFoundError: No module named 'langchain'

In [5]:
debug = False
dataset_path="triviaqa_q&a_first10.jsonl"
save_path="sc_results.jsonl"

In [4]:
model_id = "meta-llama/Llama-3.2-3B"
# model_id = "Qwen/Qwen2.5-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda", torch_dtype=torch.float16)
model = torch.compile(model)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
print(model.hf_device_map)
print(next(model.parameters()).dtype)

{'': device(type='cuda')}
torch.float16


In [6]:
def load_dataset(path: str):
    import json
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

In [7]:
dataset=load_dataset(dataset_path)

In [8]:
dataset

[{'question': 'Which American-born Sinclair won the Nobel Prize for Literature in 1930?',
  'answer': {'aliases': ['(Harry) Sinclair Lewis',
    'Harry Sinclair Lewis',
    'Lewis, (Harry) Sinclair',
    'Grace Hegger',
    'Sinclair Lewis'],
   'normalized_aliases': ['grace hegger',
    'lewis harry sinclair',
    'harry sinclair lewis',
    'sinclair lewis'],
   'matched_wiki_entity_name': '',
   'normalized_matched_wiki_entity_name': '',
   'normalized_value': 'sinclair lewis',
   'type': 'WikipediaEntity',
   'value': 'Sinclair Lewis'}},
 {'question': 'Where in England was Dame Judi Dench born?',
  'answer': {'aliases': ['Park Grove (1895)',
    'York UA',
    'Yorkish',
    'UN/LOCODE:GBYRK',
    'York, UK',
    'Eoforwic',
    'Park Grove School',
    'York Ham',
    'The weather in York',
    'City of York',
    'York, England',
    'York, Yorkshire',
    'York ham',
    'County Borough of York',
    'YORK',
    'Eoferwic',
    'Park Grove Primary School',
    'York, North Yorks

In [9]:
def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""

    def remove_articles(text):
        regex = re.compile(r"\b(a|an|the)\b", re.UNICODE)
        return re.sub(regex, " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

In [10]:
def is_correct_triviaqa(pred: str, answer_dict: dict, threshold: int = 90) -> bool:
    pred_norm = normalize_answer(pred)
    gt_norms = [normalize_answer(answer_dict["normalized_value"])] + [
        normalize_answer(a) for a in answer_dict.get("normalized_aliases", [])
    ]

    return any(pred_norm == gt for gt in gt_norms)

In [11]:
dataset[0]

{'question': 'Which American-born Sinclair won the Nobel Prize for Literature in 1930?',
 'answer': {'aliases': ['(Harry) Sinclair Lewis',
   'Harry Sinclair Lewis',
   'Lewis, (Harry) Sinclair',
   'Grace Hegger',
   'Sinclair Lewis'],
  'normalized_aliases': ['grace hegger',
   'lewis harry sinclair',
   'harry sinclair lewis',
   'sinclair lewis'],
  'matched_wiki_entity_name': '',
  'normalized_matched_wiki_entity_name': '',
  'normalized_value': 'sinclair lewis',
  'type': 'WikipediaEntity',
  'value': 'Sinclair Lewis'}}

In [ ]:
is_correct_triviaqa()

In [44]:
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=10,
    temperature=0.3,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)
llm = HuggingFacePipeline(pipeline=llm_pipeline)

The model 'OptimizedModule' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'Mamba2ForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 

In [ ]:
prompt_template = PromptTemplate(
    input_variables=["question"],
    template=SYSTEM_PROMPT,
)

parser = RegexParser(
    regex=r"Question:.*?\nAnswer:\s*(?P<answer>.+?)(?:\n|$)",  # non-greedy match
    output_keys=["answer"]
)

In [ ]:
# ========== Evaluation Logic ==========
def normalize_answer(ans):
    ans = ans.lower().strip()
    ans = ans.rstrip(". ")  # remove trailing period and space
    ans = re.sub(r"(°c|°f|degrees|percent|%)", "", ans)  # remove units
    ans = re.sub(r"[^\w\s]", "", ans)  # remove punctuation
    ans = re.sub(r"\s+", " ", ans).strip()  # normalize whitespace

    # Try to convert leading number words to digits
    match = re.match(r"^\s*(\w+)", ans)
    if match:
        try:
            ans = str(w2n.word_to_num(match.group(1)))
        except:
            pass

    return ans


def num_of_correct_answers_batch(
    predicted: List[str],
    ground_truths: List[str],
    threshold: int = 90
) -> int:
    assert len(predicted) == len(ground_truths), "Length mismatch between predictions and ground truths"

    def is_correct(pred, gt):
        pred_norm = normalize_answer(pred)
        gt_norm = normalize_answer(gt)

        # Try exact numeric comparison first
        try:
            if float(pred_norm) == float(gt_norm):
                return True
        except:
            pass

        # Fallback to fuzzy match
        full = fuzz.ratio(pred_norm, gt_norm)
        partial = fuzz.partial_ratio(pred_norm, gt_norm)
        return max(full, partial) >= threshold

    # Count correct answers in batch
    correct = sum(is_correct(p, g) for p, g in zip(predicted, ground_truths))
    return correct



def evaluate_self_consistency(questions, ground_truths, num_samples=5):
    correct_count = 0
    all_outputs = []

    for _ in range(num_samples):
        formatted_prompts = [prompt_template.format(question=q) for q in questions]
        raw_outputs = llm(formatted_prompts).strip()
        
        if debug:
            print(f"\n🔵 Raw model output:\n{raw_outputs}")

        try:
            parsed_answers = [parser.parse(o['generated_text'])['answer'].strip() for o in raw_outputs]
            # answer = parsed_answers["answer"].strip()
        except OutputParserException:
            parsed_answers = raw_outputs  # fallback to whole output
            print("⚠️  Parser failed — using raw output")

        for answer in parsed_answers:
            all_outputs.append(answer)
        if debug:
            print(f"✅ Extracted Answers: {parsed_answers}")

        if is_correct(parsed_answers, ground_truths):
            correct_count += 1

    sc_score = correct_count / num_samples
    return sc_score, all_outputs


In [47]:
def run_self_consistency_eval(
    dataset_path: str,
    num_samples: int = 5,
    debug: bool = False,
    save_path: str = None
):
    dataset = load_dataset(dataset_path)
    results = []

    for sample in tqdm(dataset):
        question = sample["question"]
        ground_truth = sample["answer"]

        sc_score, outputs = evaluate_self_consistency(
            question=question,
            ground_truth=ground_truth,
            num_samples=num_samples
        )

        result = {
            "question": question,
            "ground_truth": ground_truth,
            "self_consistency_score": sc_score,
            "generations": outputs
        }

        results.append(result)

        if debug:
            print(f"\n🔍 Question: {question}")
            print(f"🎯 Expected: {ground_truth}")
            print(f"✅ SC Score: {sc_score:.2f}")
            print("📤 Outputs:")
            for i, o in enumerate(outputs, 1):
                print(f"  {i}. {o}")

    if save_path:
        with open(save_path, "w", encoding="utf-8") as f:
            for r in results:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"\n✅ Results saved to {save_path}")

    return results


In [49]:
run_self_consistency_eval(
    dataset_path=dataset_path,
    num_samples=5,
    debug=debug,
    save_path=save_path
)

100%|██████████| 12/12 [00:11<00:00,  1.04it/s]



✅ Results saved to sc_results.jsonl


[{'question': 'What is the capital of Japan?',
  'ground_truth': 'Tokyo',
  'self_consistency_score': 1.0,
  'generations': ['Tokyo', 'Tokyo', 'Tokyo', 'Tokyo', 'Tokyo']},
 {'question': 'Who developed the theory of relativity?',
  'ground_truth': 'Albert Einstein',
  'self_consistency_score': 1.0,
  'generations': ['Albert Einstein',
   'Albert Einstein',
   'Albert Einstein',
   'Albert Einstein',
   'Albert Einstein']},
 {'question': 'What is the largest planet in our solar system?',
  'ground_truth': 'Jupiter',
  'self_consistency_score': 1.0,
  'generations': ['Jupiter', 'Jupiter', 'Jupiter', 'Jupiter', 'Jupiter']},
 {'question': 'How many continents are there?',
  'ground_truth': 'Seven',
  'self_consistency_score': 1.0,
  'generations': ['7', '7 continents', '7', '7', '7 continents.']},
 {'question': 'What is the boiling point of water at sea level in Celsius?',
  'ground_truth': '100',
  'self_consistency_score': 1.0,
  'generations': ['100 °C', '100.0', '100', '100', '100 °C']}